# Negative Binomial Model for Cyclist Volume in Adelaide

See `adelaide-model-2/data-notes.md` for discussion on decisions regarding variables and methods used.

## Variables:
- POW_CTW: N people who work in this DZN and cycle to work
- POW_FE: N people who work in this DZN and are fullly engaged in employment or study
- PD_MB: MB population density
- DNL_DZN: Distance to nearest large DZN
- M_30_40, F_15_25: Age demographics, try males 30-40 and females 15-25 (best from Dadshova's)
- H_3500: Households with a THI of > 3500, place of enumeration, MB
- INF_CR: Listed cycle routes

__Monash Bicycle Infrastructure Classification__ - Binary dummy variables for every type of bicycle infrastructure

- MBI_BP: Dedicated bike path, Shared bike path
- MBI_PP: Pedestrian path/street with cycling allowed
- MBI_PL: Protected bike lane
- MBI_BL: Buffered bike lane (kerb-side), Buffered bike lane (road-side), Buffered bike lane (both sides)
- MBI_PTL: Painted bike lane, Painted bike lane (single-side)
- MBI_Ad: Advisory bike lane, Advisory bike lane (single-side), Peak hour advisory bike lane, Peak hour advisory bike lane (single-side)
- MBI_PkPt: Peak hour painted bike lane, Peak hour painted bike lane (single-side)
- MBI_Sh: Sharrow
- MBI_SZ: Shared zone
- MBI_Bus: Bus lane with cycling allowed
- MBI_Mx: Mixed traffic
- MBI_No: Bicyclists dismount, Pedestrian path/street with cycling not allowed

## Correlations

There is bound to be a lot of correlation between many of the variables chosen for this model - eg. the Monash Classifiers. The goal of this model is to identify the best variables to use for the final model.

__*Retrieve generated graph*__

See `./GRAPH_SETUP.ipynb` for graph network generation. Graph comes with all infrastructure features from OpenStreetMap, all listed cycle routes, Mesh Blocks and Destination Zones. Mesh blocks and destination zones are loaded into __*each node*__, so MB and DZN-related variables must be calculated for (u,v) of each edge. Depending on the variable, the minimum, maximum, or average of (u,v) should be taken as the final number for that edge.

In [1]:
import osmnx as ox

graphml_path = "./graph_cache/adelaide_bike_graph.graphml"
G = ox.load_graphml(filepath=graphml_path)
print(f"Loaded graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

# Verify key attributes
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)
print(f"IS_CYCLE_ROUTE present: {'IS_CYCLE_ROUTE' in edges.columns}")

Loaded graph: 466150 nodes, 893094 edges
IS_CYCLE_ROUTE present: True


## Spatially Calculated Data

__*DNL_DZN*__

Distance to nearest large destination zone (8000+ jobs). These zones include the areas near North Terrace / Rundle / Franklin st, the RAH, between Pirie and Wakefield St, East End. 8000 is an arbitrary number.

__*PD_MB*__

Population density by mesh block

In [ ]:
import sys
sys.path.insert(0, '/Users/alexhuynh/Documents/AdelaideCyclistVolumesModel')
from scripts.distance_to_large_dzns import join_dnl_dzn

G = join_dnl_dzn(G)

Found 8 large DZNs (>= 8000 workers)
Found 3758 nodes in large DZNs
DNL_DZN stats: min=0m, max=40452m, mean=12002m
  Edges in large DZNs (dist=0): 6437/893094


In [ ]:
# import importlib
# import scripts.population_density_mbs
# importlib.reload(scripts.population_density_mbs)
from scripts.population_density_mbs import join_pd_mb

G = join_pd_mb(G)

PD_MB stats: min=0.00, max=68421.05, mean=1842.16 (per_km2)
  Edges with zero density: 202926/893094


## Census Table data:

__*POW_CTW*__

N people who work in this DZN and cycle to work

__*POW_FE*__

N people who work in this DZN and are fullly engaged in employment or study

__*M_30_40, F_15_25*__

Age demographics - males 30-40 and females 15-25

__*H_3500*__

Households with a THI of > 3500, place of enumeration, MB

In [ ]:
from scripts.join_census_cols import join_all
G = join_all(G)

## MBI Classifiers

All Monash classifiers are retrieved via `infrastructure/get_bike_infra.py`.

In [ ]:
from scripts.get_bike_infra import enrich_graph_with_monash_classifiers
G = enrich_graph_with_monash_classifiers(G)

## Misc Variables

__*INF_CR*__

Listed cycle routes

In [ ]:
edges = ox.graph_to_gdfs(G, edges=True)
for edge in edges:
    if edge['IS_CYCLE_ROUTE'] == 1: 
        edge['INF_CR'] = 1 
    else: edge['INF_CR'] = 0

# Constructing the regression matrix

In [ ]:
import pandas as pd

counts_path = "../all-data/super-tuesday-normalised/counts-2026.csv"

required_cols = ["total-count", "x", "y"]

REGRESSION_TABLE = pd.read_csv(counts_path)

missing = [col for col in required_cols if col not in REGRESSION_TABLE.columns]
if missing:
    raise ValueError(f"Missing required count columns: {missing}")

REGRESSION_TABLE = REGRESSION_TABLE[required_cols].copy()
REGRESSION_TABLE = REGRESSION_TABLE.rename(columns={"total-count": "COUNT"})
REGRESSION_TABLE["COUNT"] = pd.to_numeric(REGRESSION_TABLE["COUNT"], errors="coerce")
REGRESSION_TABLE["x"] = pd.to_numeric(REGRESSION_TABLE["x"], errors="coerce")
REGRESSION_TABLE["y"] = pd.to_numeric(REGRESSION_TABLE["y"], errors="coerce")

REGRESSION_TABLE = REGRESSION_TABLE.dropna(subset=["COUNT", "x", "y"]).reset_index(drop=True)

print(REGRESSION_TABLE.head())
print(f"Rows loaded: {len(REGRESSION_TABLE)}")

__*Join all graph attributes from nearest edge*__

# 